# A seleção que mais marcou em cada Copa

## Task 3: a seleção que mais marcou em cada Copa

## A pergunta do time de analytics

**Qual seleção mais marcou gols em cada Copa?**

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-rankings")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o código dos vídeos 2.4 e 3.1, na ordem, com as cinco views do projeto.

In [2]:
# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
        F.col("MatchesPlayed").alias("partidas_oficiais"),
        F.col("GoalsScored").alias("gols_oficiais"),
    )
)

# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

In [3]:
spark.sql("select * from resumo").show()

+----+-------------+-----+--------+-------+--------+--------+-----------+-----+
| ano|      selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+-------------+-----+--------+-------+--------+--------+-----------+-----+
|2002|     Portugal|    3|       1|      0|       2|       6|          4|    2|
|2010|      Uruguay|    7|       3|      2|       2|      11|          8|    3|
|1994|       Norway|    3|       1|      1|       1|       1|          1|    0|
|1954|  Switzerland|    4|       2|      0|       2|      11|         11|    0|
|1998|      Romania|    4|       2|      1|       1|       4|          3|    1|
|2006|      England|    5|       3|      2|       0|       6|          2|    4|
|2002|     Cameroon|    3|       1|      1|       1|       2|          3|   -1|
|2022|     Cameroon|    3|       1|      1|       1|       4|          4|    0|
|1982|        Spain|    5|       1|      2|       2|       4|          5|   -1|
|2022|United States|    4|       1|     

In [4]:
gols_por_copa = resumo.select("ano", "selecao", "gols_pro", "saldo")
gols_por_copa.createOrReplaceTempView("gols_por_copa")
gols_por_copa.orderBy("ano", F.desc("gols_pro")).show(5)

+----+-------------+--------+-----+
| ano|      selecao|gols_pro|saldo|
+----+-------------+--------+-----+
|1930|    Argentina|      18|    9|
|1930|      Uruguay|      15|   12|
|1930|United States|       7|    1|
|1930|   Yugoslavia|       7|    0|
|1930|        Chile|       5|    2|
+----+-------------+--------+-----+
only showing top 5 rows


## Por que o groupBy não basta

O `groupBy` junta as linhas de cada Copa numa só, e a seleção se perde. Uma **window function** calcula olhando o
grupo, mas **mantém todas as linhas**, com o resultado numa coluna nova.

| Parte | Pergunta | No nosso caso |
|---|---|---|
| `partitionBy` | quais linhas formam um grupo? | as seleções de uma **mesma Copa** |
| `orderBy` | em que ordem, dentro do grupo? | dos **mais gols** para os menos |

## A posição de cada seleção na sua Copa

`row_number()` numera as linhas de cada grupo: 1, 2, 3... O saldo e o nome entram no `orderBy` como desempate.

In [6]:
from pyspark.sql import Window

por_copa_mais_gols = Window.partitionBy("ano").orderBy(F.desc("gols_pro"), F.desc("saldo"), "selecao")

ranking = gols_por_copa.withColumn("posicao", F.row_number().over(por_copa_mais_gols))
ranking.filter(F.col("ano") == 1978).show(5)

+----+------------+--------+-----+-------+
| ano|     selecao|gols_pro|saldo|posicao|
+----+------------+--------+-----+-------+
|1978|   Argentina|      15|   11|      1|
|1978| Netherlands|      15|    5|      2|
|1978|      Brazil|      10|    7|      3|
|1978|West Germany|      10|    5|      4|
|1978|       Italy|       9|    3|      5|
+----+------------+--------+-----+-------+
only showing top 5 rows


In [11]:
mais_gols_por_copa = ranking.filter(F.col("posicao") == 1).orderBy("ano")
mais_gols_por_copa.show(25)

+----+------------+--------+-----+-------+
| ano|     selecao|gols_pro|saldo|posicao|
+----+------------+--------+-----+-------+
|1930|   Argentina|      18|    9|      1|
|1934|       Italy|      12|    9|      1|
|1938|     Hungary|      15|   10|      1|
|1950|      Brazil|      22|   16|      1|
|1954|     Hungary|      27|   17|      1|
|1958|      France|      23|    8|      1|
|1962|      Brazil|      14|    9|      1|
|1966|    Portugal|      17|    9|      1|
|1970|      Brazil|      19|   12|      1|
|1974|      Poland|      16|   11|      1|
|1978|   Argentina|      15|   11|      1|
|1982|      France|      16|    4|      1|
|1986|   Argentina|      14|    9|      1|
|1990|West Germany|      15|   10|      1|
|1994|      Sweden|      15|    7|      1|
|1998|      France|      15|   13|      1|
|2002|      Brazil|      18|   14|      1|
|2006|     Germany|      14|    8|      1|
|2010|     Germany|      16|   11|      1|
|2014|     Germany|      18|   14|      1|
|2018|     

## A mesma pergunta em SQL, com `QUALIFY`

`OVER (PARTITION BY ... ORDER BY ...)` é a janela; o `QUALIFY` filtra pelo resultado dela.

In [12]:
spark.sql("""
    SELECT ano, selecao, gols_pro
    FROM gols_por_copa
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ano ORDER BY gols_pro DESC, saldo DESC, selecao) = 1
    ORDER BY ano
""").show(25)

+----+------------+--------+
| ano|     selecao|gols_pro|
+----+------------+--------+
|1930|   Argentina|      18|
|1934|       Italy|      12|
|1938|     Hungary|      15|
|1950|      Brazil|      22|
|1954|     Hungary|      27|
|1958|      France|      23|
|1962|      Brazil|      14|
|1966|    Portugal|      17|
|1970|      Brazil|      19|
|1974|      Poland|      16|
|1978|   Argentina|      15|
|1982|      France|      16|
|1986|   Argentina|      14|
|1990|West Germany|      15|
|1994|      Sweden|      15|
|1998|      France|      15|
|2002|      Brazil|      18|
|2006|     Germany|      14|
|2010|     Germany|      16|
|2014|     Germany|      18|
|2018|     Belgium|      16|
|2022|      France|      16|
+----+------------+--------+



## Para consultar depois

- **`rank` e `dense_rank`:** também numeram as linhas, mas tratam empates de outro jeito. Com dois empatados em
  segundo, `rank` dá 1, 2, 2, 4 (pula a posição) e `dense_rank` dá 1, 2, 2, 3. O `row_number` sempre dá 1, 2, 3, 4.
- **`lag` e `lead`:** pegam o valor da linha anterior ou da próxima dentro do grupo, por exemplo os gols da seleção
  na Copa anterior (janela por seleção, ordenada por ano).
- **Acumulado:** `F.sum(...)` sobre uma janela com `rowsBetween(Window.unboundedPreceding, Window.currentRow)` soma do
  início do grupo até a linha atual.
- **Janela sem `orderBy`** vê o grupo inteiro: serve para colocar o total do grupo em cada linha e calcular
  percentuais.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [13]:
spark.stop()